In [2]:
import pandas as pd
import numpy as np
from datetime import timezone

INPUT_FILE = "../data stuff/newly_generated_data.csv"
OUTPUT_AUGMENTED = "../data stuff/augmented_telemetry.csv"

# Physical constants
RHO_WATER = 1000        # kg/m^3
G = 9.81               # m/s^2
CAN_AREA_M2 = 0.04     # cross-sectional area of jerry can (adjust!)

# LOAD DATA
df = pd.read_csv(INPUT_FILE)
df["recorded_at"] = pd.to_datetime(df["recorded_at"], utc=True)

df = df.sort_values(["device_id", "recorded_at"]).reset_index(drop=True)

# -----------------------------
# PRESSURE DELTA & DISPENSE RATE
df["pressure_prev"] = df.groupby("device_id")["pressure_pa"].shift(1)
df["time_prev"] = df.groupby("device_id")["recorded_at"].shift(1)

df["delta_t_hours"] = (
    df["recorded_at"] - df["time_prev"]
).dt.total_seconds() / 3600

df["pressure_delta"] = df["pressure_pa"] - df["pressure_prev"]

# Dispense rate: only when pressure decreases
df["dispense_rate_pa_per_hr"] = np.where(
    df["pressure_delta"] < 0,
    -df["pressure_delta"] / df["delta_t_hours"],
    np.nan
)

# Convert to liters/hour using hydrostatic relation
df["dispense_rate_lph"] = (
    df["dispense_rate_pa_per_hr"]
    * (CAN_AREA_M2 / (RHO_WATER * G))
    * 1000
)

# GPS QUALITY FILTER
df["valid_gps"] = (
    df["fix_type"].isin(["2D", "3D"])
    & (df["hdop"] <= 4)
    & (df["sat_count"] >= 3)
)

# WATER VOLUME DISPENSED
df["delta_volume_l"] = (
    df["pressure_delta"]
    * (CAN_AREA_M2 / (RHO_WATER * G))
    * 1000
)

df["dispensed_l"] = np.where(
    df["delta_volume_l"] < 0,
    -df["delta_volume_l"],
    0
)

df["date"] = df["recorded_at"].dt.date
daily = (
    df.groupby(["device_id", "date"])["dispensed_l"]
    .sum()
    .reset_index(name="daily_water_demand_l")
)

# SAVE OUTPUT
df.to_csv(OUTPUT_AUGMENTED, index=False)

print("Pipeline complete.")
print(f"Saved: {OUTPUT_AUGMENTED}")
print(df.head())


Pipeline complete.
Saved: ../data stuff/augmented_telemetry.csv
  device_id               recorded_at  pressure_pa  battery_v       lat  \
0   can_001 2026-01-19 00:02:26+00:00       104434       3.67  12.90337   
1   can_001 2026-01-19 00:04:38+00:00       102554       4.05  15.39732   
2   can_001 2026-01-19 00:05:06+00:00       105854       3.80  15.09193   
3   can_001 2026-01-19 00:05:57+00:00       101799       3.77  13.16758   
4   can_001 2026-01-19 00:06:21+00:00       107078       4.02  13.70753   

        lon  hdop  sat_count fix_type  speed_mps  ... pressure_prev  \
0  33.89747   0.9          9       3D       0.67  ...           NaN   
1  32.80622   1.3          9       3D       0.01  ...      104434.0   
2  33.59918   1.2         12       3D       0.08  ...      102554.0   
3  30.21720   5.4          6       2D       2.48  ...      105854.0   
4  30.79917   1.8          9       3D      11.66  ...      101799.0   

                  time_prev  delta_t_hours pressure_delta 